# Flux handwriting pilot: train a small LoRA on Hindi words, then test it

Steps: 1) check the GPU, 2) load the 40 pilot images, 3) train with ai-toolkit, 4) generate test words, 5) read them back with OCR.

Before you start: zip the folder `flux_data/words_pilot` from the repo into `words_pilot.zip` and upload it either to the Colab Files panel (left side, upload button) or to the top level of Google Drive (My Drive). Use a GPU runtime with at least 16 GB (L4 or A100 is safest; a T4 may run out of memory).

Not yet tested end to end. If a cell fails, send me the error text.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Load the pilot images
!rm -rf /content/words_pilot && mkdir -p /content/words_pilot
import os
zp = '/content/words_pilot.zip' if os.path.exists('/content/words_pilot.zip') else '/content/drive/MyDrive/words_pilot.zip'
print('using', zp)
!unzip -q -o {zp} -d /content/words_pilot_raw
import glob, os, shutil
src = os.path.dirname(glob.glob('/content/words_pilot_raw/**/manifest.csv', recursive=True)[0])
for f in glob.glob(src + '/*'): shutil.copy(f, '/content/words_pilot')
print(len(glob.glob('/content/words_pilot/*.png')), 'images,', len(glob.glob('/content/words_pilot/*.txt')), 'captions')

In [ ]:
# 3a) Install the trainer (ai-toolkit) and log in to Hugging Face for the model download
!git clone https://github.com/ostris/ai-toolkit /content/ai-toolkit
%cd /content/ai-toolkit
# Colab now runs Python 3.13, and ai-toolkit pins an old scipy that has no ready-made build for it.
# Remove that pin so pip picks a current scipy.
!sed -i -E 's/^scipy[^#]*/scipy/' requirements.txt
!pip install -q -r requirements.txt
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3b) Find the klein example config that ships with ai-toolkit and fill in our three values
import glob, yaml, os
cands = sorted(glob.glob('/content/ai-toolkit/config/examples/*klein*'))
print('example configs:', cands)

In [ ]:
# Pick the 4B LoRA example from the list above (edit the index if needed), then write our config
src = [c for c in cands if '4b' in c.lower()][0]
cfg = yaml.safe_load(open(src))
cfg['config']['name'] = 'hwdv1_words_pilot'
proc = cfg['config']['process'][0]
proc['trigger_word'] = 'HWDV1'
proc['datasets'][0]['folder_path'] = '/content/words_pilot'
proc['train']['steps'] = 1800
proc['training_folder'] = '/content/output'
# show sample prompts the trainer will render so we can judge checkpoints by eye
proc['sample']['prompts'] = [
  'HWDV1 handwritten Hindi word on white paper: खेलने',
  'HWDV1 handwritten Hindi word on white paper: नमस्ते',
  'HWDV1 handwritten Hindi word on white paper: विश्वविद्यालय',
]
# T4 (16 GB) low-memory settings. These keys exist in most ai-toolkit examples; we only set them if present.
m = proc.get('model', {})
for k in ('quantize', 'quantize_te', 'low_vram'):
    if k in m or k == 'low_vram':
        m[k] = True
proc['model'] = m
print('model block:', m)
yaml.safe_dump(cfg, open('/content/ai-toolkit/config/hwdv1_words_pilot.yaml','w'), allow_unicode=True)
print(yaml.safe_dump(cfg, allow_unicode=True)[:3000])

In [ ]:
# 3c) Train. Checkpoints and sample images land in /content/output/hwdv1_words_pilot/ every 250 steps.
%cd /content/ai-toolkit
!python run.py config/hwdv1_words_pilot.yaml

Judge the checkpoints by the sample images in `/content/output/hwdv1_words_pilot/samples`, not by the loss. Pick the step where the words look right and the hand looks natural (often 750 to 1500).

In [ ]:
# 4) Generate test words with the trained LoRA (distilled 4B model, 4 steps)
import torch, glob, os
from diffusers import Flux2KleinPipeline
pipe = Flux2KleinPipeline.from_pretrained('black-forest-labs/FLUX.2-klein-4B', torch_dtype=torch.bfloat16)
lora = sorted(glob.glob('/content/output/hwdv1_words_pilot/*.safetensors'))[-1]
print('using', lora)
pipe.load_lora_weights(lora)
pipe.enable_model_cpu_offload()  # keeps memory low on a 16 GB T4; slower but safe
words = ['घर','पानी','किताब','स्कूल','हमारा','बहुत','कृपया','विक्रेता','क्रेता','रजिस्ट्री','दस्तावेज़','संपत्ति']
os.makedirs('/content/gen', exist_ok=True)
for i, w in enumerate(words):
    img = pipe(f'HWDV1 handwritten Hindi word on white paper: {w}', num_inference_steps=4,
               generator=torch.Generator('cpu').manual_seed(i)).images[0]
    img.save(f'/content/gen/{i:02d}.png')
print('done')

In [ ]:
# 5) Read each word back with OCR and compare with what we asked for
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin > /dev/null
!pip -q install pytesseract python-Levenshtein
import pytesseract, Levenshtein
from PIL import Image
tot = ok = 0
for i, w in enumerate(words):
    got = pytesseract.image_to_string(Image.open(f'/content/gen/{i:02d}.png'), lang='hin', config='--psm 8').strip()
    cer = Levenshtein.distance(w, got) / max(len(w), 1)
    ok += cer <= 0.2; tot += 1
    print(f'{w:12s} -> {got:12s} error rate {cer:.2f}')
print(f'{ok}/{tot} words readable within 20% error')

In [ ]:
# Contact sheet to look at
from PIL import Image
ims = [Image.open(f'/content/gen/{i:02d}.png').resize((256,256)) for i in range(len(words))]
sheet = Image.new('RGB', (256*4, 256*3), 'white')
for i, im in enumerate(ims): sheet.paste(im, ((i%4)*256, (i//4)*256))
sheet.save('/content/drive/MyDrive/hwdv1_pilot_sheet.png'); sheet